# 05_CheckStage1AllOutputs: is everything needed by stage 2 present in the stage-1 collection?

Context: the stage-2 QuantumGraph (`bps_dp2cosmos_parsl_2026-11-03_stage2.yaml`, pipeline subset `stage2-recalibrate`) was built with 99773 quanta for 20 tasks, but the single `fgcmBuildFromIsolatedStar` quantum failed.

Questions:
1. Which dataset types does stage 2 read from outside itself (stage-1 outputs, calibrations, skymap, reference catalogs, schemas)?
2. Is each of them present in `<stage1 collection>` + `LSSTCam/defaults`, in the registry **and** in the datastore (files on disk)?
3. Are the stage-1 outputs complete, visit by visit and detector by detector?
4. What exactly does the failed task `fgcmBuildFromIsolatedStar` read, and does its quantum have all its inputs?

Reasoning: the QuantumGraph builder only creates a quantum whose required inputs exist in the registry, so a missing registry entry is unlikely. The usual suspects are therefore (a) datasets known to the registry whose file is absent from the datastore, (b) prerequisite inputs (reference catalog shards, look-up tables) that are optional for the graph builder but needed at run time, (c) a stage-1 run not chained into the stage-1 collection, (d) a plain run-time error, which only the log can tell.

Run with the `lsst_distrib` kernel at CC-IN2P3 (butler `main`).

- author : Sylvie Dagoret-Campagne
- creation date : 2026-11-07
- input collection : `u/dagoret/LSSTCam/runcosmos01_20261103_tract9813patchsel_RunCosmos9patches/stage1`
- pipeline : `drp_pipe` LSSTCam DRP, release `w_2026_43`, subsets `stage1-single-visit` and `stage2-recalibrate` (local copy of the expanded pipeline: `notebooks/fig_diagrams/pipeline_drp_pipe_LSSTCam_DRP.yaml`)
- stage-2 quantum diagram : `notebooks/fig_diagrams/pipeline_drp_pipe_LSSTCam_DRP_stage2-recalibrate.png`

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

SRC = Path('../../src').resolve()
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from rubincc import monster as mon

NB_TAG = '05_CheckStage1AllOutputs'
DATA_DIR = Path(f'data_{NB_TAG}')
DATA_DIR.mkdir(exist_ok=True)

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_colwidth', 90)
pd.set_option('display.width', 220)

In [ ]:
REPO = 'main'
USER = os.getenv('USER')
INSTRUMENT = 'LSSTCam'
SKYMAP = 'lsst_cells_v2'

RUN_TAG = 'runcosmos01_20261103_tract9813patchsel_RunCosmos9patches'
STAGE1 = f'u/{USER}/LSSTCam/{RUN_TAG}/stage1'   # stage-1 output = stage-2 input collection
STAGE2 = f'u/{USER}/LSSTCam/{RUN_TAG}/stage2'   # stage-2 output collection
DEFAULTS = 'LSSTCam/defaults'                   # second input collection of stage 2 (inCollection in the bps yaml)

STAGE1_LABEL = 'stage1-single-visit'
STAGE2_LABEL = 'stage2-recalibrate'
FAILED_TASK = 'fgcmBuildFromIsolatedStar'

# numbers printed by bps for the stage-2 QuantumGraph
N_VISITS_QG = 2325               # quanta of the per-visit tasks (updateVisitSummary...)
N_VISIT_DETECTORS_QG = 27807     # quanta of the per-detector tasks (refitPsfModelDetector...)

N_SAMPLE = 30   # datasets per type whose datastore record and file are checked

SCRIPTS_DIR = Path('../../scripts/myRunDRPCosmos/2026-11-03-DRPstages1and2inTractandSelectedPatches').resolve()
LOCAL_PIPELINE_YAML = Path('../fig_diagrams/pipeline_drp_pipe_LSSTCam_DRP.yaml').resolve()


def read_ids(name, default):
    path = SCRIPTS_DIR / name
    if not path.exists():
        return default
    return [int(x) for x in path.read_text().replace(',', ' ').split()]


TRACTS = read_ids('tractIds.txt', [9813])
PATCHES = read_ids('patchIds.txt', [])
DETECTORS = read_ids('ccdIds.txt', [])
print('tracts:', TRACTS)
print('patches:', PATCHES)
print('detectors:', len(DETECTORS))
print('stage-1 collection :', STAGE1)
print('stage-2 collection :', STAGE2)

## 1. Collections

Stage 2 reads `STAGE1` + `LSSTCam/defaults` (`inCollection` of the stage-2 bps yaml). The stage-1 outputs are visible through the stage-1 chained collection only if the final bps job (`butler transfer-from-graph --update-output-chain`) ran: a stage-1 run that exists but is not in the chain is reported here.

In [ ]:
butler = mon.get_butler(REPO, collections=[STAGE1, DEFAULTS])
reg = butler.registry


def chain_of(name):
    try:
        if reg.getCollectionType(name).name == 'CHAINED':
            return list(reg.getCollectionChain(name))
    except Exception:
        pass
    return [name]


for coll in (STAGE1, STAGE2, DEFAULTS):
    try:
        ctype = reg.getCollectionType(coll).name
    except Exception as exc:
        print(f'{coll}: NOT FOUND ({type(exc).__name__})')
        continue
    members = chain_of(coll) if ctype == 'CHAINED' else []
    print(f'{coll}: {ctype}, {len(members)} members')
    for member in members[:10]:
        print('    ->', member)

stage1_chain = chain_of(STAGE1)
stage1_runs = sorted(reg.queryCollections(f'{STAGE1}/*'))
print()
print(len(stage1_runs), 'stage-1 run collection(s) found under', STAGE1)
for run in stage1_runs:
    flag = 'in the chain' if run in stage1_chain else 'NOT IN THE CHAIN (final job not done?)'
    print('   ', run, '->', flag)

## 2. What does stage 2 read?

The pipeline graphs of `stage1-single-visit` and `stage2-recalibrate` are built from the pipeline yaml and resolved against the registry. A dataset type read by a stage-2 task and not produced inside stage 2 is an *external input*. If a stage-1 task produces it, it must be in the stage-1 collection; otherwise it must come from `LSSTCam/defaults` (calibrations, skymap, reference catalogs...). Regular inputs, init-inputs (schemas) and prerequisite inputs (reference catalogs, look-up tables) are all included.

In [ ]:
from lsst.pipe.base import Pipeline

# bps used ${DRP_PIPE_DIR}/pipelines/LSSTCam/DRP.yaml#<label>: use it if available, else the local expanded copy
drp_pipe_dir = os.getenv('DRP_PIPE_DIR')
if drp_pipe_dir and (Path(drp_pipe_dir) / 'pipelines/LSSTCam/DRP.yaml').exists():
    PIPELINE_FILE = Path(drp_pipe_dir) / 'pipelines/LSSTCam/DRP.yaml'
else:
    PIPELINE_FILE = LOCAL_PIPELINE_YAML
print('pipeline file:', PIPELINE_FILE)


def load_graph(label):
    pipeline = Pipeline.from_uri(f'{PIPELINE_FILE}#{label}')
    return pipeline.to_graph(registry=reg)


def produced_names(pg):
    names = set()
    for node in pg.tasks.values():
        names.update(edge.parent_dataset_type_name for edge in node.outputs.values())
        names.update(edge.parent_dataset_type_name for edge in node.init.outputs.values())
    return names


def read_edges(label, node):
    rows_e = []
    groups = (('input', node.inputs), ('prerequisite', node.prerequisite_inputs), ('init-input', node.init.inputs))
    for kind, mapping in groups:
        for connection, edge in mapping.items():
            rows_e.append(dict(task=label, kind=kind, connection=connection,
                               dataset_type=edge.parent_dataset_type_name,
                               minimum=getattr(edge, 'minimum', np.nan)))
    return rows_e


pg1 = load_graph(STAGE1_LABEL)
pg2 = load_graph(STAGE2_LABEL)
print(f'{STAGE1_LABEL}: {len(pg1.tasks)} tasks')
print(f'{STAGE2_LABEL}: {len(pg2.tasks)} tasks (the bps QuantumGraph reports 20 tasks)')

produced1 = produced_names(pg1)
produced2 = produced_names(pg2)

all_reads = pd.DataFrame([row for label, node in pg2.tasks.items() for row in read_edges(label, node)])
all_reads['minimum'] = pd.to_numeric(all_reads['minimum'], errors='coerce')
external = all_reads[~all_reads.dataset_type.isin(produced2)].copy()

summary = (external.groupby('dataset_type')
           .agg(kinds=('kind', lambda s: ','.join(sorted(set(s)))),
                minimum=('minimum', 'min'),
                consumers=('task', lambda s: ', '.join(sorted(set(s)))))
           .reset_index())
summary['origin'] = np.where(summary.dataset_type.isin(produced1), 'stage1', 'defaults')
print()
print(summary.origin.value_counts().to_string())
display(summary[['dataset_type', 'origin', 'kinds', 'minimum', 'consumers']])

## 3. Check every external input

- **stage-1 outputs**: number of datasets in the stage-1 collection, number of runs they come from, and registry-versus-datastore consistency on a random sample of `N_SAMPLE` datasets per type (`stored_many` = datastore record, `URI.exists()` = file on disk);
- **inputs from `LSSTCam/defaults`**: at least one dataset; for reference catalogs, every HTM shard overlapping the tract(s) must exist (a missing shard is invisible to the graph builder but fatal at run time);
- `expected` is the number of quanta of the stage-2 QuantumGraph for visit-level (2325) or visit-detector-level (27807) datasets: a difference means stage 1 changed after the graph was built, or some stage-1 quanta failed.

Status values: `OK`, `MISSING` (absent and required), `OPTIONAL (absent)`, `PARTIAL COVERAGE`, `FILES MISSING` (in the registry, not in the datastore), `NOT REGISTERED`, `QUERY ERROR`.

In [ ]:
def count_datasets(name, collections, where='', bind=None):
    # find-first fails for calibration-like types in CALIBRATION collections: retry without it
    last = ''
    for find_first in (True, False):
        try:
            res = reg.queryDatasets(name, collections=collections, findFirst=find_first, where=where, bind=bind)
            return res.count(exact=True), ('' if find_first else 'findFirst=False')
        except Exception as exc:
            last = f'{type(exc).__name__}: {str(exc)[:150]}'
    return -1, last


def defaults_where(dims):
    parts, bind = [], {}
    if 'instrument' in dims:
        parts.append('instrument = inst')
        bind['inst'] = INSTRUMENT
    if 'skymap' in dims:
        parts.append('skymap = sm')
        bind['sm'] = SKYMAP
    return ' AND '.join(parts), bind


def expected_count(dims):
    if 'tract' in dims or 'patch' in dims or 'visit' not in dims:
        return np.nan
    return N_VISIT_DETECTORS_QG if 'detector' in dims else N_VISITS_QG


def stored_check(refs, n_sample=N_SAMPLE, seed=0):
    # datastore record (stored_many) and physical file (URI exists) for a random sample of datasets
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(refs), size=min(n_sample, len(refs)), replace=False)
    sample = [refs[i] for i in idx]
    n_not_stored, n_no_file = -1, -1
    try:
        n_not_stored = sum(1 for ok in butler.stored_many(sample).values() if not ok)
    except Exception as exc:
        print('   stored_many failed:', type(exc).__name__, str(exc)[:100])
    try:
        get_uri = getattr(butler, 'get_uri', None) or butler.getURI
        n_no_file = sum(1 for ref in sample if not get_uri(ref).exists())
    except Exception as exc:
        print('   URI check failed:', type(exc).__name__, str(exc)[:100])
    return len(sample), n_not_stored, n_no_file


try:
    skymap = butler.get('skyMap', skymap=SKYMAP)
except Exception as exc:
    skymap = None
    print('skymap not found, HTM coverage of reference catalogs not checked:', type(exc).__name__)


def htm_ids_for_tracts(level):
    import lsst.sphgeom as sphgeom
    pixelization = sphgeom.HtmPixelization(level)
    ids = set()
    for tract in TRACTS:
        region = skymap[tract].outer_sky_polygon
        for lo, hi in pixelization.envelope(region).ranges():
            ids.update(range(lo, hi))
    return sorted(ids)

In [ ]:
rows = []
ids_by_type = {}   # {stage-1 dataset type: set of (visit[, detector])}, used in section 5

for rec in summary.itertuples():
    name = rec.dataset_type
    row = dict(dataset_type=name, origin=rec.origin, dims='', n_found=-1, expected=np.nan, n_runs=np.nan,
               sample=np.nan, not_stored=np.nan, no_file=np.nan, status='', note='',
               minimum=rec.minimum, kinds=rec.kinds, consumers=rec.consumers)
    try:
        dtype = reg.getDatasetType(name)
    except Exception:
        row['status'] = 'NOT REGISTERED'
        rows.append(row)
        continue
    dims = set(dtype.dimensions.names)
    row['dims'] = ','.join(sorted(dims))
    row['expected'] = expected_count(dims) if rec.origin == 'stage1' else np.nan
    partial = False
    try:
        if rec.origin == 'stage1':
            refs = list(reg.queryDatasets(name, collections=[STAGE1], findFirst=True))
            row['n_found'] = len(refs)
            row['n_runs'] = len({ref.run for ref in refs})
            if refs:
                row['sample'], row['not_stored'], row['no_file'] = stored_check(refs)
                if 'visit' in dims:
                    key = ('visit', 'detector') if 'detector' in dims else ('visit',)
                    ids_by_type[name] = {tuple(ref.dataId[k] for k in key) for ref in refs}
        else:
            htm = [d for d in dims if d.startswith('htm')]
            if htm and skymap is not None:
                needed = htm_ids_for_tracts(int(htm[0][3:]))
                res = reg.queryDatasets(name, collections=[DEFAULTS], findFirst=True,
                                        where=f'{htm[0]} IN (ids)', bind={'ids': needed})
                found = {ref.dataId[htm[0]] for ref in res}
                row['n_found'] = len(found)
                row['note'] = f'{len(found)}/{len(needed)} {htm[0]} shards over tract(s) {TRACTS}'
                partial = len(found) < len(needed)
            else:
                where, bind = defaults_where(dims)
                row['n_found'], row['note'] = count_datasets(name, [DEFAULTS], where, bind)
    except Exception as exc:
        row['note'] = f'{type(exc).__name__}: {str(exc)[:150]}'

    n = row['n_found']
    if n < 0:
        row['status'] = 'QUERY ERROR'
    elif n == 0:
        row['status'] = 'OPTIONAL (absent)' if rec.minimum == 0 else 'MISSING'
    elif partial:
        row['status'] = 'PARTIAL COVERAGE'
    elif row['not_stored'] > 0 or row['no_file'] > 0:
        row['status'] = 'FILES MISSING'
    else:
        row['status'] = 'OK'
    exp = row['expected']
    if not np.isnan(exp) and n != exp:
        row['note'] += f' count differs from QuantumGraph ({n} vs {int(exp)})'
    rows.append(row)
    print(f'{row["status"]:18s} {name}  n={n}')

status_df = pd.DataFrame(rows).sort_values(['origin', 'dataset_type']).reset_index(drop=True)

## 4. Verdict

In [ ]:
cols = ['dataset_type', 'origin', 'dims', 'n_found', 'expected', 'n_runs', 'sample', 'not_stored', 'no_file', 'status', 'note']
display(status_df[cols])

n_ok = int((status_df.status == 'OK').sum())
print(f'{len(status_df)} external dataset types read by {STAGE2_LABEL}: {n_ok} OK')
problems = status_df[status_df.status != 'OK']
if problems.empty:
    print('Everything needed by stage 2 is present in', STAGE1, '+', DEFAULTS)
else:
    print('PROBLEMS:')
    display(problems[['dataset_type', 'origin', 'status', 'consumers', 'note']])

status_df.to_csv(DATA_DIR / 'external_inputs_status.csv', index=False)

## 5. Completeness of the stage-1 outputs

For the visit-level and visit-detector-level dataset types read by stage 2, which (visit, detector) pairs are present in some types and absent from others? A pair that has `single_visit_star_footprints` but no `preliminary_visit_image` (or the reverse) points to a stage-1 quantum that failed or was never transferred.

In [ ]:
def completeness(groups):
    union = set().union(*groups.values())
    rows_c = [dict(dataset_type=n, n_present=len(s), n_missing=len(union - s), first_missing=sorted(union - s)[:3])
              for n, s in sorted(groups.items())]
    return union, pd.DataFrame(rows_c)


vd_groups = {n: s for n, s in ids_by_type.items() if s and len(next(iter(s))) == 2}
v_groups = {n: {t[0] for t in s} for n, s in ids_by_type.items() if s}

union_vd, df_vd = completeness(vd_groups)
union_v, df_v = completeness(v_groups)

print(f'(visit, detector) pairs: union = {len(union_vd)}  (stage-2 QuantumGraph: {N_VISIT_DETECTORS_QG})')
display(df_vd)
print(f'visits: union = {len(union_v)}  (stage-2 QuantumGraph: {N_VISITS_QG})')
display(df_v)

## 6. What does the failed task read?

All connections of `fgcmBuildFromIsolatedStar` with the status of each dataset type. `prerequisite` inputs (reference catalog, fgcm look-up table) are the ones the graph builder may let through even if absent.

In [ ]:
node = pg2.tasks[FAILED_TASK]
print('task class :', getattr(node, 'task_class_name', '?'))
print('dimensions :', getattr(node, 'dimensions', '?'))
print('outputs    :', sorted(e.parent_dataset_type_name for e in node.outputs.values()))

status_by_type = status_df.set_index('dataset_type').status
reads = all_reads[all_reads.task == FAILED_TASK].copy()
reads['status'] = reads.dataset_type.map(status_by_type).fillna('made inside stage 2')
display(reads[['connection', 'dataset_type', 'kind', 'minimum', 'status']])
display(status_df[status_df.consumers.str.contains(FAILED_TASK)][cols])

## 7. (optional) The failed quantum in the stage-2 QuantumGraph

Best effort: loads the QuantumGraph file saved by bps in the submit directory and checks, for each input of the `fgcmBuildFromIsolatedStar` quantum, how many datasets have no datastore record. The file format and the API depend on the release: if the load fails, set `QGRAPH_FILE` by hand or skip this section.

In [ ]:
SUBMIT_ROOT = Path(f'/sps/lsst/users/{USER}/bps/submit')
QGRAPH_FILE = None   # set a path by hand to override the automatic search

if QGRAPH_FILE is None:
    base = SUBMIT_ROOT / STAGE2
    found_files = []
    if base.exists():
        for pattern in ('*/*.qgraph', '*/*.qg'):
            found_files += list(base.glob(pattern))
    found_files = sorted(found_files, key=lambda p: p.stat().st_mtime)
    QGRAPH_FILE = found_files[-1] if found_files else None
print('QuantumGraph file:', QGRAPH_FILE)

if QGRAPH_FILE is not None:
    try:
        from lsst.pipe.base import QuantumGraph

        qg = QuantumGraph.loadUri(str(QGRAPH_FILE))
        task_def = qg.findTaskDefByLabel(FAILED_TASK)
        quanta = [n.quantum for n in qg.getNodesForTask(task_def)]
        print(len(quanta), 'quantum/quanta for', FAILED_TASK)
        quantum = quanta[0]
        rows_q = []
        for kind, mapping in (('input', quantum.inputs), ('init-input', quantum.initInputs)):
            for dtype, refs in mapping.items():
                refs = list(refs) if isinstance(refs, (list, tuple)) else [refs]
                stored = butler.stored_many(refs)
                rows_q.append(dict(kind=kind, dataset_type=dtype.name, n_refs=len(refs),
                                   n_not_stored=sum(1 for ok in stored.values() if not ok),
                                   runs=sorted({r.run for r in refs})[:2]))
        display(pd.DataFrame(rows_q))
    except Exception as exc:
        print('could not inspect the QuantumGraph:', type(exc).__name__, str(exc)[:300])

## 8. (optional) Log of the failed quantum

If every input is present, the failure is a run-time error and only the log tells which. This looks in the stage-2 submit directory for files whose name contains the task label and prints the end of the most recent ones. If nothing is found, paste the traceback from the parsl logs.

In [ ]:
base = SUBMIT_ROOT / STAGE2
log_hits = []
if base.exists():
    log_hits = sorted((p for p in base.rglob(f'*{FAILED_TASK}*') if p.is_file()), key=lambda p: p.stat().st_mtime)
print(len(log_hits), 'file(s) with', FAILED_TASK, 'in their name under', base)
for path in log_hits[-3:]:
    print('=====', path)
    lines = path.read_text(errors='replace').splitlines()
    print(*lines[-40:], sep=chr(10))